# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Task type: ranking. I want to put pages in order, so the team reviews the pages most likely to be declining first. A ranking fits because the team has limited time and can only fix a few pages.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [6]:
import pandas as pd

URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(URL)

df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(df["is_declining"].value_counts())
print("Share declining:", round(df["is_declining"].mean(), 3))

is_declining
1    16262
0    13738
Name: count, dtype: int64
Share declining: 0.542


Target: is the page declining. This is a proxy, because I cannot measure "needs a refresh" directly. The label is trend_direction equal to "down", which means the page dropped compared with the previous month. It comes from a defined rule on observed numbers, not from opinion. I will not use trend_pct as an input, because the label is built from it. That would be leakage.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [8]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining"].values

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
hand_rule = stale * visible * df["impressions_90d"]

random_scores = np.random.RandomState(42).rand(len(df))

print("Random   Precision@50:", round(precision_at_k(random_scores, y, 50), 3))
print("Hand rule Precision@50:", round(precision_at_k(hand_rule, y, 50), 3))

Random   Precision@50: 0.56
Hand rule Precision@50: 0.68


Metric: Precision@50. Of the 50 pages my ranking puts first, how many are really declining. I chose it because the team can only fix a few pages, so only the top of the list matters. A random pick is right about 54% of the time, because 54.2% of all pages are declining. So a good ranking must score clearly above 0.542. My simple hand rule scored 0.680 in the earlier notebook, so I will compare against that. This is decision-support only.

Unit of analysis: one row is one page. The column content_id names the page, and client_id says which client owns it. The other columns are facts about that page, such as impressions, clicks, position, age and word count. The target column is_declining is 1 if the page trend is down and 0 if not. The ranking will put these pages in order.

In [9]:
cols = ["content_id", "client_id", "impressions_90d", "clicks_90d",
        "avg_position", "content_age_days", "days_since_last_update",
        "word_count", "is_declining"]

print("One row = one page. Rows:", len(df))
print("Each page appears once:", df["content_id"].is_unique)
df[cols].head(10)

One row = one page. Rows: 30000
Each page appears once: True


,content_id,client_id,impressions_90d,clicks_90d,avg_position,content_age_days,days_since_last_update,word_count,is_declining
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,187,20,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,445,25,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,141,20,3515.0,1
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,463,22,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,263,14,2803.0,1
5,content_d4084a4bc775,client_f369cb89fc,3970,1,8.5,147,20,3080.0,1
6,content_9a34b442b552,client_8722616204,20,0,7.0,90,20,3059.0,1
7,content_a63219c6e95a,client_19581e27de,1724,1,21.2,445,22,NaN,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,46.0,90,20,3807.0,1
9,content_c27558df2b0c,client_19581e27de,1240,2,4.9,257,104,NaN,1


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule uses cut points I guessed, such as 180 days and 500 impressions. Pages are messier than that. Age, impressions, position and clicks work together, and a guessed cut point can miss a page that is close to the line. A model can test many cut points on 30,000 pages and keep the best ones, and a small tree can still be read. I must be careful: in my earlier test the hand rule scored 0.680 and a shallow tree scored 0.600. A deeper tree scored 0.720, but on the same data it learned from. So the claim is directional. I need a train and test split to see if the model really beats the rule.

In [11]:
flagged = hand_rule > 0

print("Pages the rule can rank:", int(flagged.sum()), "of", len(df))
print("Declining pages the rule scores zero:", int(((~flagged) & (df["is_declining"] == 1)).sum()))
print("Share declining among zero-score pages:", round(df.loc[~flagged, "is_declining"].mean(), 3))


Pages the rule can rank: 17 of 30000
Declining pages the rule scores zero: 16246
Share declining among zero-score pages: 0.542


I tested the hand rule for a weakness. It can rank only 17 of 30,000 pages, and it gives a score of zero to 16,246 declining pages. Those zero-score pages are declining 54.2% of the time, the same as any random page, so the rule says nothing about them. This means my 0.680 score comes from 17 sharp picks plus 33 near-random picks. A model could keep ranking where the rule stops. This is observed in this sample and directional only.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.